# Spark SQL Example

In [1]:
from SparkUtils import SparkUtils
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, TimestampType, FloatType
from pyspark.sql.functions import col
from datetime import datetime

In [2]:
master_url = "spark://spark-master:7077"
app_name = "Example 01 - Spark SQL"

spark = SparkUtils(master_url, app_name)._spark

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 00:39:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
data = [
    (1, "Alice", 29),
    (2, "Bob", 35),
    (3, "Charlie", 41)
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df = spark.createDataFrame(data, schema)

df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)



In [4]:
factory_data = [
    ("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
    ("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
    ("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
    ("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
    ("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
    ("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
    ("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
    ("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
    ("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
    ("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

factory_schema = StructType([
    StructField("id", StringType(), True),
    StructField("date", TimestampType(), True),
    StructField("temperature", FloatType(), True)
])

factory_df = spark.createDataFrame(factory_data, factory_schema)

factory_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- temperature: float (nullable = true)



In [5]:
print(f"Number of rows: {factory_df.count()}")

Number of rows: 10


In [6]:
filtered_factory_df = factory_df.filter(col("temperature") > 75)

filtered_factory_df.show(truncate=False)

+----+-------------------+-----------+
|id  |date               |temperature|
+----+-------------------+-----------+
|M001|2026-01-26 08:00:00|75.3       |
|M001|2026-01-26 08:10:00|76.1       |
|M001|2026-01-26 08:25:00|77.5       |
|M001|2026-01-26 08:40:00|78.0       |
+----+-------------------+-----------+



In [7]:
record_count = factory_df.count()

print(f"Total records: {record_count}")

Total records: 10


In [8]:
ordered_df = factory_df.orderBy(col("temperature"), ascending=False)

ordered_df.show()

[Stage 8:=============================>                             (1 + 1) / 2]

+----+-------------------+-----------+
|  id|               date|temperature|
+----+-------------------+-----------+
|M001|2026-01-26 08:40:00|       78.0|
|M001|2026-01-26 08:25:00|       77.5|
|M001|2026-01-26 08:10:00|       76.1|
|M001|2026-01-26 08:00:00|       75.3|
|M003|2026-01-26 08:45:00|       74.6|
|M003|2026-01-26 08:30:00|       73.2|
|M003|2026-01-26 08:15:00|       72.4|
|M002|2026-01-26 08:35:00|       70.1|
|M002|2026-01-26 08:20:00|       69.8|
|M002|2026-01-26 08:05:00|       68.7|
+----+-------------------+-----------+



In [9]:
grouped_df = factory_df.groupBy("id").count()

grouped_df.show()

[Stage 9:=============================>                             (1 + 1) / 2]

+----+-----+
|  id|count|
+----+-----+
|M002|    3|
|M003|    3|
|M001|    4|
+----+-----+



In [10]:
from pyspark.sql.functions import avg, min, max

agg_df = factory_df.groupBy("id").agg(
    avg("temperature").alias("avg_temp"),
    min("temperature").alias("min_temp"),
    max("temperature").alias("max_temp")
)

agg_df.show()

[Stage 12:=============================>                            (1 + 1) / 2]

+----+-----------------+--------+--------+
|  id|         avg_temp|min_temp|max_temp|
+----+-----------------+--------+--------+
|M002|69.53333282470703|    68.7|    70.1|
|M003|73.39999898274739|    72.4|    74.6|
|M001|76.72500038146973|    75.3|    78.0|
+----+-----------------+--------+--------+



In [12]:
spark.stop()